**Fecha de creación**: 08/06/2026

**Lenguaje de programación**: Python 3.12.13

**Descripción del programa**:
Se aplica la técnica de regresión logística de acuerdo al artículo de "Introduction to Logistic Regression" por Moo K. Chung. Se utilizó el conjunto de datos descargado [UC Irvine Machine Learning Repository](https://https://archive.ics.uci.edu/dataset/426/autism+screening+adult), correspondientes a respuestas de Adultos que contestaron el instrumento de evaluación Q10 (Cociente de Espectro Autista). El instrumento traducido al español puede revisarse en [autism research centre](https://www.autismresearchcentre.com/content/uploads/2024/11/AQ10-Adult_Spanish.pdf). Se ajustan tres modelos con 11, 5 y 2 grados de libertad, encontrándose el modelo de 5 grados de libertad como el más útil.

In [1]:
#@title Librerías
import pandas as pd
import numpy as np
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,roc_auc_score
from scipy.stats import chi2

# Funciones

## Modelo

In [2]:
#Separación de datos para posterior validación cruzada, se uso una semilla para hacer reproducibles los resultados
def separacion(vindep,vdep,tamEntrenamiento,semilla,dummies=False):
  #estructura selectiva para el caso de variables categóricas
  if dummies==True:
    X_train, X_test, y_train, y_test,train_idx, test_idx  = train_test_split(
        vindep.values,
        vdep.values,
        train_size   = tamEntrenamiento,
        random_state = semilla,
        #El argumento stratify clasifica vdep en la misma proporcion que el set de entrenamiento para evitar un test de prueba lleno de 0 o 1
        stratify     = vdep,
        shuffle      = True)
    dummies=pd.get_dummies(vindep, prefix=vindep.name, drop_first=True, dtype=int)
    X_train=dummies.iloc[train_idx]
    X_test=dummies.iloc[test_idx]
  else:
    X_train, X_test, y_train, y_test = train_test_split(
      vindep.values,
      vdep.values,
      train_size   = tamEntrenamiento,
      random_state = semilla,
      shuffle      = True)
  #Al no contar con una muestra proporcionada de casos positivos y negativos se calcula la propoción de datos para evitar erros en la clasificación
  umbral=y_train.mean().item()
  return X_train, X_test, y_train, y_test, umbral

In [3]:
#Logaritmo de verosimilitud con clip en pi para evitar indefiniciones
def log_likelihood(X, y, beta):
    xb  = X @ beta
    pi  = np.exp(xb) / (1 + np.exp(xb))
    # Clip para evitar log(0)
    pi = np.clip(pi, 1e-30, 1 - 1e-30)
    ll = np.sum(y * xb + np.log(1 - pi))
    return ll

In [4]:
#Log-likelihood ratio test
def L_Ratio(ll_simple,ll_completo):
  return -2*(ll_simple-ll_completo)

In [5]:
#Estimación de log-likelihood del modelo nulo
def modelo_nulo(X,y):
    n = X.shape[0]
    # --- Modelo nulo: solo intercepto ---
    X_nulo        = np.ones((n, 1))          # sin hstack, ya ES el intercepto
    beta_nulo     = np.zeros((1, 1))
    gnorm         = 1
    while gnorm > 0.001:
        xb        = X_nulo @ beta_nulo
        pi        = np.exp(xb) / (1 + np.exp(xb))
        g         = X_nulo.T @ (y - pi)
        gnorm     = np.linalg.norm(g)
        s_diag    = (pi * (1 - pi)).ravel()
        H         = X_nulo.T @ (s_diag[:, None] * X_nulo)
        beta_nulo = beta_nulo + np.linalg.pinv(H) @ g

    ll_nulo = log_likelihood(X_nulo, y, beta_nulo)
    return ll_nulo

In [6]:
#Regresion Logística adaptando el código de matlab del artículo de Chung (2020)
def RL(X, y,tabla=False):
    X=X.astype(float)
    y=y.astype(float).reshape(-1,1)
    n, *resto = X.shape
    if resto==[]:
      k=1
    else:
      k=resto[0]
    # Se añade una columna de unos para el intersección
    X = np.hstack([np.ones((n, 1)), X])

    # Beta se inicializa en ceros con el tamaño de la cantidad de columnas de X +1
    beta = np.zeros((k + 1, 1))
    gnorm = 1

    #Estimación de coeficientes beta
    while gnorm > 0.001:
        # Cálculo de probabilidades, el @ es para multiplicar matrices
        extb = np.exp(X @ beta)
        pi = extb / (1 + extb)

        # Gradiente: dirección de máximo crecimiento
        g = X.T @ (y - pi)
        gnorm = np.linalg.norm(g)

        # Varianza de una distribución Bernoulli
        s_diag = (pi * (1 - pi)).ravel()
        S = np.diag(s_diag) # Matriz de pesos para el Hessiano

        H = X.T @ S @ X # Hessiano: matriz de segundas derivadas

        beta = beta + np.linalg.pinv(H) @ g # Actualización de Newton-Raphson

    # Cálculo de log-likelihood ratio test entre el modelo propuesto y la hipótesis nula
    ll = log_likelihood(X, y, beta)
    ll_nulo = modelo_nulo(X,y)
    LR       = L_Ratio(ll_nulo,ll)
    #p-value usando scipy.stats
    p_valor  = chi2.sf(LR, df=k)   # sf = 1 - cdf (cola derecha)

    #Impresión de resultados con formato
    if tabla==True:
        sig      = "p_valor < 0.001" if p_valor < 0.001 else "p_valor < 0.01" if p_valor < 0.01 else "p_valor < 0.05" if p_valor < 0.05 else "p_valor < 0.1"  if p_valor < 0.1  else ""
        print("  Datos del modelo\n")
        print(f"  {'GL':<3}{'Ll_Ratio':>11} {'p-valor':>9}  {'Sig.':>19}")
        print("  " + "-" * 65)
        print(f"  {k+1:<2}{LR:>10.3f}{p_valor:>26}  {sig:>16}")

    # Regresa los valores de los coeficientes, log-likelihood del modelo, log-likelihood ratio y el p-value respecto a la hipótesis nula
    return beta, ll,LR, p_valor

In [7]:
# Estimación de la contribución de cada variable independiente
def lrt_vars(X, y,beta,ll,nombres_vars=None):
    #Formateo de datos para evitar errores en el proceso
    ll=float(ll)
    if hasattr(X, "values"):
      X = X.values
    if hasattr(y, "values"):
      y = y.values
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, *resto = X.shape
    if resto==[]:
      k=1
      X = X.reshape(-1, 1)
    else:
      k=resto[0]

    #Inicializar arreglo de resultados
    resultados = []

    #Encabezado de los datos
    print(f"  Likelihood Ratio por variables\n")
    print(f"  {'Variable':<12}{'Coef':>7}{'LR stat':>10}{'p-valor':>12}  Sig.")
    print("  " + "-" * 55)

    #Si no hay nombres de columnas por defecto se imprimirán como X_subíndice
    if nombres_vars is None:
       nombres_vars = [f"X{i+1}" for i in range(k)]

    for j in range(k):
        # Modelo sin la variable j
        X_reducido      = np.delete(X, j, axis=1)   # elimina columna j
        if X_reducido.shape[1] == 0:
            X_reducido = np.ones((n, 1))  # Columna de unos para el modelo reducido
        _,lred,_,_ = RL(X_reducido, y)
        lred=float(lred)
        LR_j=L_Ratio(lred,ll)
        p_j      = chi2.sf(LR_j, df=1)
        sig      = "***" if p_j < 0.001 else "**" if p_j < 0.01 else "*" if p_j < 0.05 else "."  if p_j < 0.1  else "!" if p_j>=0.1 else ""

        # Se imprimen los nombres de las variables independientes, los valores respectivos de beta, el log-likelihood ratio test, el p-value respecto al modelo sin esa variable y una comparación visual de la significancia
        print(f"  {nombres_vars[j]:<12}{beta[j].item():>8.4f}{LR_j:>10.4f} {p_j:>10.6f}  {sig:>2}")
        #resultados.append((nombres_vars[j], LR_j, p_j))

    #Leyenda de la comparación visual de la significancia estadística
    print("\n  *** p<0.001  ** p<0.01  * p<0.05  . p<0.1  ! p>=0.1")

## Evaluación

In [8]:
#Cálculo de la probabilidad pi y la clasificación (pi binarizada) del modelo de acuerdo a un umbral
def predecir(X,beta,umbral_ajustado):
  X=X.astype(float)
  n, k = X.shape
  X=np.hstack([np.ones((n, 1)), X]) #Se añade la intersección
  extb = np.exp(X @ beta)
  pi = extb / (1 + extb) #distirbución de probabilidad
  predicciones=np.where(pi<umbral_ajustado, 0, 1) #Predicciones al binarizar pi
  return pi.flatten(),predicciones.flatten()

In [9]:
# Cpalculo del estadístico Q de press según el artículo de Chung 2020
def poder_discriminante(y_test, y_pred):
    # Convertir a arreglos de numpy si no lo son
    y_pred = np.array(y_pred)

    # Si la clasificacion es correcto e_i = 0 si no e_i=1
    errores = np.where(y_test != y_pred, 1, 0).flatten()

    # Tasa de error gamma
    n = len(y_test) #Tamaño de la muestra
    tasa_error = np.sum(errores) / n

    poder_discriminante = 1 - tasa_error
    #Estadísitco Q de Press para evaluar la capacidad predictiva
    Q_press= n*(((2*poder_discriminante)-1)**2)
    return Q_press

In [10]:
#Impresión de métricas de discriminación: Precisión y Área bajo la curva ROC con sklearn.metric, además del estadístico Q de Press
def validacion(X,beta,y_test,umbral_ajustado):
  _,preds=predecir(X,beta,umbral_ajustado) #Se calculan las predicciones
  Q_Press=poder_discriminante(y_test,preds)
  precision = accuracy_score(y_test, preds)
  roc = roc_auc_score(y_test, preds)
  #Impresión de encabezado con formato
  print("  Datos de validación y discriminación\n")
  print(f"{'Precision':>11} {'Area roc':>9}{'Q de Press':>12}")
  print("  " + "-" * 65)
  #Impresión de resultados
  print(f"{precision:>7.3f}{roc:>9.3f}{Q_Press:>24}\n")

In [11]:
# Comparación de un modelos más simple LLS con un modelo más complejos LLC
def comparar(LLS,LLC,betaS,betaC,nombres):
  LRmods=L_Ratio(LLS,LLC) #Likelihood-ratio test
  p_value = chi2.sf(LRmods, df=len(betaC)-len(betaS)) #Cálculo del p-value
  #Impresión de encabezado
  print('  Comparación '+nombres)
  print("  " + "-" * 55)
  #Impresión de valores
  print(f'  LLRT: {LRmods:.6f}    p-value:{p_value:.6f}\n')

# Data Set: Cuestionario Q10

In [24]:
#@title Leer datos y explorar data frame
df = arff.loadarff('/content/Autism-Adult-Data.arff')
df = pd.DataFrame(df[0])

#Dar formato a los datos
df = df.map(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)
#Descartar columnas que no son de interés
df=df.drop(columns=['age','gender','ethnicity','jundice','contry_of_res','used_app_before','age_desc','relation','Class/ASD'])
df.rename(columns={'austim': 'autismo'}, inplace=True) #Se renombra para evitar typos
df['autismo']=df['autismo'].map({'yes': 1, 'no': 0}) #Se pasan a 1 y 0 los 'yes'/'no'

#Impresión de aspectos relevantes del data frame
print(f'Sujetos con {df.autismo.value_counts().to_string()}\n') #Conteo de sujetos con y sin diagnóstico de TEA
print(f'Datos faltantes\n{df.isnull().sum()}') #Verificar que no hay columnas con datos faltantes
print(f'\nEstadísticos descriptivos\n{df.describe().round(3)}') #Impresión de estadísticos descriptivo con redondeo a 3 cifras

Sujetos con autismo
0    613
1     91

Datos faltantes
A1_Score     0
A2_Score     0
A3_Score     0
A4_Score     0
A5_Score     0
A6_Score     0
A7_Score     0
A8_Score     0
A9_Score     0
A10_Score    0
autismo      0
result       0
dtype: int64

Estadísticos descriptivos
       autismo   result
count  704.000  704.000
mean     0.129    4.875
std      0.336    2.501
min      0.000    0.000
25%      0.000    3.000
50%      0.000    4.000
75%      0.000    7.000
max      1.000   10.000


## Modelo 1 (Todas las preguntas como variables independientes) - 11 grados de libertad

In [14]:
#Segmentación de los datos
X_train1,X_test1,y_train1,y_test1,umbral1=separacion(df.drop(columns=['autismo','result']),df['autismo'],0.8,123)

In [15]:
#Estimación del modelo de regresión lineal usando MLE y validación con el subconjunto de prueba
beta1,LL1,LR1,pval1=RL(X_train1,y_train1,True)
print('\n\n')
validacion(X_test1,beta1,y_test1,umbral1)

  Datos del modelo

  GL    Ll_Ratio   p-valor                 Sig.
  -----------------------------------------------------------------
  11    35.638     9.710896612965449e-05   p_valor < 0.001



  Datos de validación y discriminación

  Precision  Area roc  Q de Press
  -----------------------------------------------------------------
  0.610    0.723       6.815602836879432



In [16]:
# Contribución de cada variable
lrt_vars(X_train1,y_train1,beta1,LL1,df.drop(columns=['autismo','result']).columns)

  Likelihood Ratio por variables

  Variable       Coef   LR stat     p-valor  Sig.
  -------------------------------------------------------
  A1_Score     -3.2644    3.8769   0.048956   *
  A2_Score      0.6559    0.4455   0.504457   !
  A3_Score      0.1796    0.0641   0.800121   !
  A4_Score     -0.0742   10.2704   0.001352  **
  A5_Score      0.9758    0.0009   0.975550   !
  A6_Score     -0.0094    0.1272   0.721393   !
  A7_Score     -0.1163    4.8619   0.027455   *
  A8_Score     -0.6032    0.0314   0.859362   !
  A9_Score      0.0487    2.2582   0.132911   !
  A10_Score     0.4832    3.4098   0.064812   .

  *** p<0.001  ** p<0.01  * p<0.05  . p<0.1  ! p>=0.1


## Modelo 2 (Las 4 preguntas más significativas según el modelo 1) - 5 grados de libertad

In [17]:
#Segmentacion de los datos
X_train2,X_test2,y_train2,y_test2,umbral2=separacion(df[['A1_Score','A4_Score','A7_Score','A10_Score']],df['autismo'],0.8,123)

In [18]:
#Estimación del modelo logit
beta2,LL2,LR2,pval2=RL(X_train2,y_train2,True)
print('\n\n')
validacion(X_test2,beta2,y_test2,umbral2)

  Datos del modelo

  GL    Ll_Ratio   p-valor                 Sig.
  -----------------------------------------------------------------
  5     32.305    1.6572131939479886e-06   p_valor < 0.001



  Datos de validación y discriminación

  Precision  Area roc  Q de Press
  -----------------------------------------------------------------
  0.631    0.706        9.70921985815603



In [19]:
#Evaluación de contribución de las variables
lrt_vars(X_train2,y_train2,beta2,LL2,df[['A1_Score','A4_Score','A7_Score','A10_Score']].columns)

  Likelihood Ratio por variables

  Variable       Coef   LR stat     p-valor  Sig.
  -------------------------------------------------------
  A1_Score     -3.1977    4.7786   0.028816   *
  A4_Score      0.7098   15.6671   0.000076  ***
  A7_Score      1.0616    4.5995   0.031982   *
  A10_Score    -0.5711    5.0708   0.024331   *

  *** p<0.001  ** p<0.01  * p<0.05  . p<0.1  ! p>=0.1


## Modelo 3 (Suma del puntaje de las 10 preguntas) - 2 grados de libertad

In [20]:
#Entrenamiento de los datos
X_train3,X_test3,y_train3,y_test3,umbral3=separacion(df['result'],df['autismo'],0.8,123)

In [21]:
#Estimación modelo logit
beta3,LL3,LR3,pval3=RL(X_train3.reshape(-1,1),y_train3,True)
print('\n\n')
validacion(X_test3.reshape(-1,1),beta3,y_test3,umbral3)

  Datos del modelo

  GL    Ll_Ratio   p-valor                 Sig.
  -----------------------------------------------------------------
  2     17.000     3.738521299057415e-05   p_valor < 0.001



  Datos de validación y discriminación

  Precision  Area roc  Q de Press
  -----------------------------------------------------------------
  0.638    0.710      10.787234042553196



In [22]:
#Contribución de las variables
lrt_vars(X_train3.reshape(-1,1),y_train3,beta3,LL3)

  Likelihood Ratio por variables

  Variable       Coef   LR stat     p-valor  Sig.
  -------------------------------------------------------
  X1           -2.9241   16.9997   0.000037  ***

  *** p<0.001  ** p<0.01  * p<0.05  . p<0.1  ! p>=0.1


## Comparación de los modelos

In [23]:
#Cálculo de Likelihood Ratio Test y p-value
comparar(LL2,LL1,beta2,beta1,'M_5gl y M_11gl')
comparar(LL3,LL1,beta3,beta1,'M_2gl y M_11gl')
comparar(LL3,LL2,beta2,beta1,'M_2gl y M_5gl')

  Comparación M_5gl y M_11gl
  -------------------------------------------------------
  LLRT: 3.333321    p-value:0.765997

  Comparación M_2gl y M_11gl
  -------------------------------------------------------
  LLRT: 18.638631    p-value:0.028447

  Comparación M_2gl y M_5gl
  -------------------------------------------------------
  LLRT: 15.305310    p-value:0.018010

